In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Module 4: Business Applications of Machine Learning

This module connects ML methods to real business problems in three key domains:
Marketing, Finance, and Operations.

## Section 4.1 — ML in Marketing

**Objectives**

By the end of this section you will be able to:

- Apply K-Means clustering to segment customers using RFM data.
- Use the elbow method to select an appropriate number of clusters.
- Build and evaluate a churn prediction model using AUC-ROC.
- Translate model outputs into actionable marketing strategies.

> **Cooking analogy:** A great restaurant does not serve every guest the same
> fixed menu — it learns their preferences and personalises the experience. A
> guest who always orders fish gets a different recommendation than one who loves
> red meat. Marketing ML does exactly this: it studies each customer's ordering
> history to group similar guests, predict who is about to leave, and decide
> which dish (offer) to put in front of each person at exactly the right moment.

Marketing generates rich customer data that ML can turn into competitive
advantages: personalised offers, targeted campaigns, and churn prevention.

### Customer Segmentation with K-Means

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

np.random.seed(10)
n = 500

df_mkt = pd.DataFrame({
    "Recency_Days"   : np.random.randint(1, 365, n),    # days since last purchase
    "Frequency"      : np.random.randint(1, 30, n),      # purchases per year
    "Monetary_Value" : np.random.lognormal(7, 1, n).round(2),  # avg spend
})

scaler = StandardScaler()
X_sc   = scaler.fit_transform(df_mkt)

In [ ]:
# Elbow method — choose optimal number of clusters
inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sc)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertia, "bo-")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia (Within-cluster Sum of Squares)")
ax.set_title("Elbow Method for Optimal k")
ax.axvline(x=4, color="red", linestyle="--", label="Elbow at k=4")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with k=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
df_mkt["Segment"] = km4.fit_predict(X_sc)

# Profile each segment
seg_profile = df_mkt.groupby("Segment").agg(
    Avg_Recency   = ("Recency_Days",   "mean"),
    Avg_Frequency = ("Frequency",      "mean"),
    Avg_Monetary  = ("Monetary_Value", "mean"),
    Count         = ("Recency_Days",   "count")
).round(1)

print("Customer Segment Profiles (RFM):")
print(seg_profile)

In [ ]:
# Label segments based on RFM profile
seg_labels = {
    seg_profile["Avg_Monetary"].idxmax()  : "Champions",
    seg_profile["Avg_Recency"].idxmax()   : "At-Risk",
}

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(df_mkt["Recency_Days"], df_mkt["Monetary_Value"],
                     c=df_mkt["Segment"], cmap="tab10", alpha=0.5, s=20)
ax.set_xlabel("Recency (Days Since Last Purchase)")
ax.set_ylabel("Monetary Value ($)")
ax.set_title("Customer Segments — RFM Clustering")
plt.colorbar(scatter, ax=ax, label="Segment")
plt.tight_layout()
plt.show()

### Churn Prediction Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay

np.random.seed(7)
n = 1000
df_churn2 = pd.DataFrame({
    "Recency_Days"    : np.random.randint(1, 365, n),
    "Frequency"       : np.random.randint(1, 50, n),
    "Avg_Spend"       : np.random.lognormal(4, 0.8, n).round(2),
    "Email_Opens"     : np.random.randint(0, 30, n),
    "NPS_Score"       : np.random.randint(1, 11, n),
})
df_churn2["Churned"] = (
    (df_churn2["Recency_Days"] > 180) & (df_churn2["NPS_Score"] < 6)
).astype(int)

X = df_churn2.drop(columns="Churned")
y = df_churn2["Churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            random_state=42, stratify=y)
rf_mkt = RandomForestClassifier(n_estimators=200, random_state=42)
rf_mkt.fit(X_tr, y_tr)

y_prob = rf_mkt.predict_proba(X_te)[:, 1]
print(f"AUC-ROC: {roc_auc_score(y_te, y_prob):.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(rf_mkt, X_te, y_te, ax=ax, name="RF Churn Model")
ax.set_title("ROC Curve — Churn Prediction")
plt.tight_layout()
plt.show()

---

### Student Task 4.1

Your marketing manager wants to design targeted email campaigns for different customer groups.

1. Using `df_mkt`, try **k = 3** and **k = 5** clusters. Which do you prefer? Why?
2. For each segment, write a brief **marketing strategy** (1–2 sentences) recommending how to engage that customer group.
3. Using the churn model probabilities, create a DataFrame of the **top 50 customers** most likely to churn. What action would you recommend for each?

In [ ]:
# Your code here

---

### Evaluation Questions 4.1

1. **RFM** in customer analytics stands for:
   a) Revenue, Frequency, Market
   b) Recency, Frequency, Monetary (correct)
   c) Return, Function, Model
   d) Risk, Forecast, Margin

2. The "elbow" in a K-Means elbow plot indicates:
   a) The maximum number of clusters allowed
   b) The point where adding more clusters yields diminishing returns (correct)
   c) An error in the data
   d) The optimal feature count

3. K-Means is an example of which type of learning?
   a) Supervised learning
   b) Reinforcement learning
   c) Semi-supervised learning
   d) Unsupervised learning (correct)

4. AUC-ROC of 0.5 indicates:
   a) Perfect classification
   b) 50 % accuracy
   c) Model is no better than random guessing (correct)
   d) 50 % of customers will churn

5. A **false negative** in churn prediction means:
   a) Predicting a customer will churn when they will not
   b) Correctly identifying a loyal customer
   c) Predicting a customer will stay when they actually churn (correct)
   d) Correctly predicting a churner

---